# 01 · Contract 계층 — ToolSpec → Catalog → DSL/tools 렌더링 → 파싱·검증

> **이 노트북의 목표**
> 1. Ganglion에서 "모델이 보는 것"과 "실행기가 받는 것" 사이를 잇는 **Contract 계층(Module 3)** 을 바닥부터 직접 만들어 본다.
> 2. 리포트 `docs/poc_verification_report.md` §14.2의 **카탈로그 크기 표(1.58x / 2.69x / 3.40x)** 를 이 자리에서 재현한다.
> 3. 리포트 §6의 **데이터셋 500건 분포**를 재현하면서 `expected`가 로드 시점에 검증된다는 사실을 확인한다.
>
> **필요한 것**: GPU 없음, API 키 없음. 전부 결정론적이고 수 초 안에 끝난다.

## 왜 Contract부터인가

전체 파이프라인의 케이스당 데이터 흐름은 이렇다.

```
user prompt ──▶ ModelClient.invoke() ──▶ JSON DSL 문자열
                                              │
                                              ▼
                              Catalog.parse_json_dsl()  ◀── Contract 계층 (이 노트북)
                                              │
                                              ▼
                                         ActionPlan  ──▶ plan == expected  (exact match)
                                              │
                                              ▼
                                   emit_tool_calls() ──▶ 실행기
```

LLM이 하는 일은 **문자열 하나를 만드는 것**뿐이다. 그 문자열을 안전한 tool call로 바꾸는 것, 정답과 비교하는 것, 그리고 모델에게 보여줄 프롬프트를 만드는 것은 전부 Contract 계층이 결정론적으로 한다. 이후 노트북(SFT, post-correction, BFCL)에서 등장하는 모든 숫자는 결국 이 계층의 `parse_json_dsl`과 `ActionPlan.__eq__`가 정의한다. 그래서 여기를 먼저 손으로 만들어 본다.

**규칙**: 이 노트북은 라이브러리 코드를 복사하지 않는다. 항상 `ganglion.*`을 import하고, 구현이 궁금하면 `inspect.getsource`로 들여다본다. 노트북은 설명과 실험 결과만 가진다.

In [1]:
# 0. 준비 — 저장소 루트를 찾아 sys.path와 cwd를 맞춘다 (notebooks/ 밑에서 실행되므로)
from __future__ import annotations

import inspect
import json
import os
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() and (p / "ganglion").is_dir():
            return p
    raise RuntimeError("pyproject.toml + ganglion/ 를 가진 상위 디렉토리를 찾지 못했습니다")


REPO = _find_repo_root(Path.cwd())
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))


def show(obj, first: int | None = None) -> None:
    # 라이브러리 소스를 그 자리에서 열어본다. 복사하지 않기 위한 도구.
    src = inspect.getsource(obj)
    lines = src.splitlines()
    if first is not None:
        lines = lines[:first] + [f"    ... ({len(src.splitlines()) - first} more lines)"]
    print(f"# {inspect.getsourcefile(obj)}")
    print("\n".join(lines))


print("repo root:", REPO)
print("python   :", sys.version.split()[0])

repo root: /home/kist/workspace/ganglion
python   : 3.12.14


## 1. ArgSpec — 인자 하나를 어떻게 선언하는가

`ganglion/contract/tool_spec.py`에는 인자 타입이 7종 있다. 전부 `frozen=True` dataclass다.

| ArgSpec | 용도 | 정규화 훅 |
|---|---|---|
| `EnumArg` | 고정 선택지 | `aliases` (`"거실" → "living"`), `bool_true/bool_false` |
| `IntArg` | 정수 + 범위 | `allow_percent` (`"70%" → 70`) |
| `NumberArg` | 실수 + 범위 | — |
| `StringArg` | 자유 문자열 | `aliases`, `pattern` |
| `BoolArg` | 불리언 | `"yes"/"on"/"1"` 등 허용 |
| `TimeArg` | `HH:MM` 24시간 | 형식·시간 범위 검사 |
| `RawArg` | JSON Schema 조각을 그대로 | 중첩 배열 등 위 타입으로 표현 못 하는 형태 |

핵심은 **`aliases`** 다. 모델이 `"거실"`이라고 써도, `"living room"`이라고 써도 validator가 `"living"` 하나로 접는다. 이것이 리포트에서 말하는 *semantic-normalized exact match* 의 "normalized"가 뜻하는 바다.

In [2]:
from ganglion.contract import (
    ActionPlan, BoolArg, Catalog, DSLValidationError, EnumArg, IntArg,
    NumberArg, RawArg, StringArg, TimeArg, ToolCall, ToolSpec,
)

show(EnumArg)
show(IntArg)

# /home/kist/workspace/ganglion/ganglion/contract/tool_spec.py
@dataclass(frozen=True)
class EnumArg:
    values: tuple[str, ...]
    aliases: Mapping[str, str] = field(default_factory=dict)
    required: bool = True
    description: str = ""
    bool_true: str | None = None
    bool_false: str | None = None
    kind: str = field(init=False, default="enum")
# /home/kist/workspace/ganglion/ganglion/contract/tool_spec.py
@dataclass(frozen=True)
class IntArg:
    min_value: int | None = None
    max_value: int | None = None
    required: bool = True
    description: str = ""
    allow_percent: bool = False
    kind: str = field(init=False, default="integer")


### 1.1 장난감 카탈로그를 처음부터 만들어 보기

실제 `iot_light_5`를 보기 전에, 도구 2개짜리 카탈로그를 직접 만든다. 선풍기 도구다.

In [3]:
ROOM = EnumArg(
    values=("living", "bedroom"),
    aliases={"거실": "living", "living room": "living", "침실": "bedroom", "방": "bedroom"},
)
FAN_STATE = EnumArg(values=("on", "off"), aliases={"켜": "on", "꺼": "off"},
                    bool_true="on", bool_false="off")
SPEED = IntArg(min_value=1, max_value=5, required=False)

toy_tools = (
    ToolSpec(
        name="set_fan",
        description="Turn a room fan on/off and optionally set speed 1..5.",
        args=(("room", ROOM), ("state", FAN_STATE), ("speed", SPEED)),
    ),
    ToolSpec(
        name="get_fan_state",
        description="Read the fan state of a room.",
        args=(("room", ROOM),),
    ),
)

toy = Catalog(
    name="toy_fan_2",
    tools=toy_tools,
    examples=(("거실 선풍기 3단으로 켜줘",
               '{"calls":[{"action":"set_fan","args":{"room":"living","state":"on","speed":3}}]}'),),
    extra_rules=("Use canonical English room names.",),
)
toy

Catalog(name='toy_fan_2', tools=(ToolSpec(name='set_fan', description='Turn a room fan on/off and optionally set speed 1..5.', args=(('room', EnumArg(values=('living', 'bedroom'), aliases={'거실': 'living', 'living room': 'living', '침실': 'bedroom', '방': 'bedroom'}, required=True, description='', bool_true=None, bool_false=None, kind='enum')), ('state', EnumArg(values=('on', 'off'), aliases={'켜': 'on', '꺼': 'off'}, required=True, description='', bool_true='on', bool_false='off', kind='enum')), ('speed', IntArg(min_value=1, max_value=5, required=False, description='', allow_percent=False, kind='integer'))), dsl_args_override=None, custom_validator=None, defaults_when_missing=(), strip_unknown_args=False, prompt_correction=None), ToolSpec(name='get_fan_state', description='Read the fan state of a room.', args=(('room', EnumArg(values=('living', 'bedroom'), aliases={'거실': 'living', 'living room': 'living', '침실': 'bedroom', '방': 'bedroom'}, required=True, description='', bool_true=None, bool_

## 2. 하나의 source of truth에서 두 표현을 렌더링한다

`Catalog`는 같은 `ToolSpec` 튜플에서 두 가지를 만든다.

- `render_json_dsl()` — DSL 경로에서 **시스템 프롬프트에 붙는 짧은 텍스트**
- `render_openai_tools()` — native 경로에서 **`tools=[...]` 파라미터로 넘기는 JSON Schema**

이 둘이 같은 곳에서 나오기 때문에 "DSL vs native" 비교가 apples-to-apples가 된다. 리포트의 모든 토큰 절감 수치는 이 두 함수의 출력 길이 차이에서 시작한다.

In [4]:
dsl_text = toy.render_json_dsl()
print(dsl_text)

Return JSON only.
JSON shape: {"calls":[{"action":"<action>","args":{...}}]}
Allowed actions:
- set_fan args {"room": "living"|"bedroom", "state": "on"|"off", "speed": optional integer 1..5}
- get_fan_state args {"room": "living"|"bedroom"}
Rules:
- Use canonical English room names.
- Do not include explanations or Markdown.
Examples:
User: 거실 선풍기 3단으로 켜줘
JSON: {"calls":[{"action":"set_fan","args":{"room":"living","state":"on","speed":3}}]}



In [5]:
native_tools = toy.render_openai_tools()
print(json.dumps(native_tools, indent=2, ensure_ascii=False))

[
  {
    "type": "function",
    "function": {
      "name": "set_fan",
      "description": "Turn a room fan on/off and optionally set speed 1..5.",
      "parameters": {
        "type": "object",
        "properties": {
          "room": {
            "type": "string",
            "enum": [
              "living",
              "bedroom"
            ]
          },
          "state": {
            "type": "string",
            "enum": [
              "on",
              "off"
            ]
          },
          "speed": {
            "type": "integer",
            "minimum": 1,
            "maximum": 5
          }
        },
        "additionalProperties": false,
        "required": [
          "room",
          "state"
        ]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "get_fan_state",
      "description": "Read the fan state of a room.",
      "parameters": {
        "type": "object",
        "properties": {
          "room": {
            "ty

In [6]:
native_text = json.dumps(native_tools, ensure_ascii=False)
print(f"DSL chars    : {len(dsl_text):>5}")
print(f"native chars : {len(native_text):>5}")
print(f"native / DSL : {len(native_text) / len(dsl_text):.2f}x")

DSL chars    :   445
native chars :   679
native / DSL : 1.53x


도구 2개짜리에서도 native가 더 길다. 이유를 코드에서 직접 확인하자. DSL은 도구당 **한 줄**이고, native는 도구당 **JSON Schema 객체 전체**다. 특히 `room` enum이 native에서는 도구마다 반복되지만 DSL에서는 `"living"|"bedroom"` 짧은 표기로 끝난다.

In [7]:
from ganglion.contract import catalog as catalog_mod

show(catalog_mod.Catalog.render_json_dsl)
show(catalog_mod._render_dsl_arg_value)

# /home/kist/workspace/ganglion/ganglion/contract/catalog.py
    def render_json_dsl(self) -> str:
        lines: list[str] = [
            "Return JSON only.",
            'JSON shape: {"calls":[{"action":"<action>","args":{...}}]}',
        ]
        if self.allow_empty_calls:
            lines.append('If no tool call is needed, return exactly {"calls":[]}.')
        lines.append("Allowed actions:")
        for tool in self.tools:
            args_text = tool.dsl_args_override or _render_dsl_args(tool.args)
            lines.append(f"- {tool.name} args {args_text}")
        rules = list(self.extra_rules) + [
            "Do not include explanations or Markdown.",
        ]
        lines.append("Rules:")
        lines.extend(f"- {rule}" for rule in rules)
        if self.examples:
            lines.append("Examples:")
            for prompt, response in self.examples:
                lines.append(f"User: {prompt}")
                lines.append(f"JSON: {response}")
        return "\n".

## 3. parse_json_dsl — 문자열이 ActionPlan이 되기까지

모델 출력(문자열)이 들어오면 순서대로 이렇게 처리된다.

1. `json.loads` — 실패하면 `DSLValidationError("invalid JSON…")`
2. `"calls"` 배열인지 확인 (또는 top-level `"action"` 하나)
3. 각 call마다 `validate_call`:
   - `action`이 카탈로그에 있는가
   - (post-correction: `defaults_when_missing` → `strip_unknown_args` → `prompt_correction`) — **5번 노트북**에서 다룸
   - `_validate_flat_args`: 선언되지 않은 인자 거부, 필수 인자 확인, 타입별 정규화
4. 불변 `ToolCall`들의 튜플인 `ActionPlan` 반환

먼저 정상 경로.

In [8]:
raw = '{"calls":[{"action":"set_fan","args":{"room":"거실","state":true,"speed":"3"}}]}'
plan = toy.parse_json_dsl(raw)
plan

ActionPlan(calls=(ToolCall(action='set_fan', args={'room': 'living', 'state': 'on', 'speed': 3}),))

세 가지 정규화가 한 번에 일어났다. `"거실"` → `"living"` (alias), `true` → `"on"` (`bool_true`), `"3"` → `3` (문자열 정수 허용). 모델이 이 셋 중 어떤 형태로 내놓아도 **같은 ActionPlan** 이 된다. 이게 exact match를 "raw 문자열 비교"가 아니라 "정규화 후 값 비교"로 만드는 장치다.

이제 실패 경로를 전부 밟아 본다. 각 오류 메시지는 4번 노트북의 repair loop가 모델에게 되돌려주는 텍스트이기도 하다.

In [9]:
bad_inputs = {
    "invalid JSON":        '{"calls":[{"action":"set_fan"',
    "unknown action":      '{"calls":[{"action":"set_light","args":{"room":"living","state":"on"}}]}',
    "missing required":    '{"calls":[{"action":"set_fan","args":{"room":"living"}}]}',
    "unknown arg":         '{"calls":[{"action":"set_fan","args":{"room":"living","state":"on","color":"red"}}]}',
    "enum out of set":     '{"calls":[{"action":"set_fan","args":{"room":"garage","state":"on"}}]}',
    "int out of range":    '{"calls":[{"action":"set_fan","args":{"room":"living","state":"on","speed":9}}]}',
    "empty calls":         '{"calls":[]}',
    "args not object":     '{"calls":[{"action":"get_fan_state","args":"living"}]}',
}
for label, raw in bad_inputs.items():
    try:
        toy.parse_json_dsl(raw)
        print(f"{label:<18} -> (unexpectedly OK)")
    except DSLValidationError as e:
        print(f"{label:<18} -> DSLValidationError: {e}")

invalid JSON       -> DSLValidationError: invalid JSON: Expecting ',' delimiter
unknown action     -> DSLValidationError: unsupported action: set_light
missing required   -> DSLValidationError: set_fan.state is required
unknown arg        -> DSLValidationError: set_fan: unknown arg 'color'
enum out of set    -> DSLValidationError: unsupported room: garage
int out of range   -> DSLValidationError: speed must be <= 5
empty calls        -> DSLValidationError: 'calls' must not be empty
args not object    -> DSLValidationError: call.args must be an object


`"empty calls"` 가 오류라는 점을 기억해 두자. 기본값 `allow_empty_calls=False` 에서는 "아무 도구도 부르지 않는다"가 **표현 불가능**하다. BFCL `irrelevance` 카테고리에서 DSL 경로가 처음에 native보다 12pp 뒤진 원인이 정확히 이것이고, M5에서 `allow_empty_calls=True` 를 추가해 역전시켰다. 아래 §7에서 직접 확인한다.

In [10]:
show(catalog_mod._validate_flat_args)
show(catalog_mod._normalize_enum)

# /home/kist/workspace/ganglion/ganglion/contract/catalog.py
def _validate_flat_args(tool: ToolSpec, args: dict[str, Any]) -> dict[str, Any]:
    if not tool.args:
        if args:
            raise DSLValidationError(f"{tool.name} does not accept args")
        return {}

    declared = {name for name, _ in tool.args}
    for key in args:
        if key not in declared:
            raise DSLValidationError(f"{tool.name}: unknown arg '{key}'")

    normalized: dict[str, Any] = {}
    for name, spec in tool.args:
        present = name in args and args[name] is not None
        if not present:
            if spec.required:
                raise DSLValidationError(f"{tool.name}.{name} is required")
            continue
        normalized[name] = _normalize_value(name, spec, args[name])
    return normalized
# /home/kist/workspace/ganglion/ganglion/contract/catalog.py
def _normalize_enum(name: str, spec: EnumArg, raw: Any) -> str:
    if isinstance(raw, bool):
        if spec.bool_true is

## 4. ActionPlan 값 동등성 = exact match 메트릭

`ActionPlan`과 `ToolCall`은 frozen dataclass라서 `==`가 **값 비교**다. 평가기의 `exact_match`는 문자 그대로 `result.plan == expected` 다. 인자 순서, 공백, 따옴표 스타일은 전혀 영향을 주지 않는다. 오직 정규화 후의 `(action, args)` 만 본다.

In [11]:
from ganglion.contract.types import ActionPlan as _AP  # 소스 확인용
show(_AP)

expected = ActionPlan(calls=(ToolCall(action="set_fan", args={"room": "living", "state": "on", "speed": 3}),))

variants = [
    '{"calls":[{"action":"set_fan","args":{"room":"living","state":"on","speed":3}}]}',
    '{"calls":[{"action":"set_fan","args":{"speed":"3","state":true,"room":"living room"}}]}',
    '{ "calls" : [ { "action" : "set_fan" , "args" : { "room":"거실", "state":"켜", "speed":3 } } ] }',
    '{"action":"set_fan","args":{"room":"living","state":"on","speed":3}}',   # top-level single call
    '{"calls":[{"action":"set_fan","args":{"room":"living","state":"on","speed":4}}]}',  # speed 다름
]
for v in variants:
    print(toy.parse_json_dsl(v) == expected, " <-", v)

# /home/kist/workspace/ganglion/ganglion/contract/types.py
@dataclass(frozen=True)
class ActionPlan:
    calls: tuple[ToolCall, ...]

    def to_jsonable(self) -> dict[str, Any]:
        return {
            "calls": [
                {"action": call.action, "args": call.args}
                for call in self.calls
            ]
        }
True  <- {"calls":[{"action":"set_fan","args":{"room":"living","state":"on","speed":3}}]}
True  <- {"calls":[{"action":"set_fan","args":{"speed":"3","state":true,"room":"living room"}}]}
True  <- { "calls" : [ { "action" : "set_fan" , "args" : { "room":"거실", "state":"켜", "speed":3 } } ] }
True  <- {"action":"set_fan","args":{"room":"living","state":"on","speed":3}}
False  <- {"calls":[{"action":"set_fan","args":{"room":"living","state":"on","speed":4}}]}


## 5. 실제 카탈로그: `iot_light_5`

이제 프로젝트가 실제로 쓰는 카탈로그를 연다. `ganglion/contract/builtins/iot_light.py` 가 모듈 레벨 `CATALOG`를 export하고, `get_catalog(tier)` 가 세 tier의 레지스트리다.

도구 5개: `list_devices`, `get_light_state`, `set_light`, `schedule_light`, `create_scene`. 이 중 `create_scene`은 `RawArg` + `custom_validator`로 **중첩 호출**(scene 안의 `set_light` 배열)을 검증하는 유일한 도구다.

In [12]:
from ganglion.contract.builtins import TIERS, get_catalog

iot = get_catalog("iot_light_5")
for t in iot.tools:
    req = ", ".join(t.required_arg_names()) or "-"
    opt = ", ".join(n for n, s in t.args if not s.required) or "-"
    print(f"{t.name:<16} required=[{req}]  optional=[{opt}]"
          f"{'  (custom_validator)' if t.custom_validator else ''}")

list_devices     required=[-]  optional=[-]
get_light_state  required=[room]  optional=[-]
set_light        required=[room, state]  optional=[brightness, color_temp]
schedule_light   required=[room, at, state]  optional=[brightness]
create_scene     required=[name, actions]  optional=[-]  (custom_validator)


In [13]:
print(iot.render_json_dsl())

Return JSON only.
JSON shape: {"calls":[{"action":"<action>","args":{...}}]}
Allowed actions:
- list_devices args {}
- get_light_state args {"room": one of living, bedroom, kitchen, hallway, office}
- set_light args {"room": one of living, bedroom, kitchen, hallway, office, "state": "on"|"off", "brightness": optional integer 0..100, "color_temp": optional "warm"|"neutral"|"cool"}
- schedule_light args {"room": one of living, bedroom, kitchen, hallway, office, "at": "HH:MM" 24h time, "state": "on"|"off", "brightness": optional integer 0..100}
- create_scene args {"name": one of movie, relax, focus, sleep, "actions": array of set_light calls}
Rules:
- Use canonical English room names.
- Use 24-hour HH:MM for schedules.
- Do not include explanations or Markdown.
Examples:
User: 거실 불 70%로 켜줘
JSON: {"calls":[{"action":"set_light","args":{"room":"living","state":"on","brightness":70}}]}
User: 밤 10시 반에 침실 조명 꺼줘
JSON: {"calls":[{"action":"schedule_light","args":{"room":"bedroom","at":"22:30","

위 텍스트가 **모델이 보는 카탈로그 전부**다. 리포트가 "compact catalog와 예시만 전달"이라고 말하는 그것이다. 이 텍스트는 `ganglion/lm/prompts.py`의 시스템 프롬프트 템플릿에 끼워진다. 그 템플릿은 SFT 학습 데이터와 **바이트 단위로 동일**해야 한다는 불변식이 있다(4번 노트북에서 다시 본다).

In [14]:
from ganglion.lm.prompts import SYSTEM_PROMPT_TEMPLATE, _dsl_messages

show(_dsl_messages)
msgs = _dsl_messages(iot, "거실 불 70%로 켜줘")
print("--- system (앞 3줄) ---")
print("\n".join(msgs[0]["content"].splitlines()[:3]), "...")
print("--- user ---")
print(msgs[1]["content"])

# /home/kist/workspace/ganglion/ganglion/lm/prompts.py
def _dsl_messages(catalog: Catalog, user_prompt: str) -> list[dict[str, Any]]:
    return [
        {
            "role": "system",
            "content": SYSTEM_PROMPT_TEMPLATE.format(dsl=catalog.render_json_dsl()),
        },
        {"role": "user", "content": user_prompt},
    ]
--- system (앞 3줄) ---
You convert user requests into the JSON DSL below. The response must be valid JSON.

Return JSON only. ...
--- user ---
거실 불 70%로 켜줘


In [15]:
# 같은 카탈로그의 native 표현 — set_light 하나만 보자
set_light_native = next(t for t in iot.render_openai_tools() if t["function"]["name"] == "set_light")
print(json.dumps(set_light_native, indent=2, ensure_ascii=False))

{
  "type": "function",
  "function": {
    "name": "set_light",
    "description": "Turn a room light on or off and optionally set brightness or color temperature.",
    "parameters": {
      "type": "object",
      "properties": {
        "room": {
          "type": "string",
          "enum": [
            "living",
            "bedroom",
            "kitchen",
            "hallway",
            "office"
          ]
        },
        "state": {
          "type": "string",
          "enum": [
            "on",
            "off"
          ]
        },
        "brightness": {
          "type": "integer",
          "minimum": 0,
          "maximum": 100
        },
        "color_temp": {
          "type": "string",
          "enum": [
            "warm",
            "neutral",
            "cool"
          ]
        }
      },
      "additionalProperties": false,
      "required": [
        "room",
        "state"
      ]
    }
  }
}


같은 `set_light`가 DSL에서는 한 줄(`- set_light args {...}`)이었다. native에서 `room`의 enum 5개, `color_temp`의 enum 3개가 전부 펼쳐지고, `schedule_light`에서도 `room` enum이 또 한 번 반복된다. 도구가 50개가 되면 이 반복이 어떻게 되는지가 다음 절이다.

## 6. 재현 ①: 카탈로그 크기 스케일링 (리포트 §14.2)

리포트 `docs/poc_verification_report.md` §14.2 표:

| Tier | Tools | DSL chars | Native chars | Native/DSL |
|---|---:|---:|---:|---:|
| iot_light_5 | 5 | 1,307 | 2,062 | 1.58x |
| home_iot_20 | 20 | 2,525 | 6,796 | 2.69x |
| smart_home_50 | 50 | 4,643 | 15,795 | 3.40x |

같은 측정을 `ganglion.benchmarks.iot.scaling.measure` 로 지금 실행해서 비교한다. 이 표는 오프라인·결정론적이므로 정확히 일치해야 한다. (일치하지 않으면 그 사이 카탈로그가 바뀐 것이고, 리포트가 갱신 대상이다.)

In [16]:
from ganglion.benchmarks.iot.scaling import measure

REPORTED = {  # docs/poc_verification_report.md §14.2 (2026-04-28)
    "iot_light_5":   (1307, 2062),
    "home_iot_20":   (2525, 6796),
    "smart_home_50": (4643, 15795),
}

print(f"{'tier':<14}{'tools':>6}{'dsl':>7}{'native':>8}{'ratio':>7}   {'reported (ratio)':<20}{'Δdsl':>6}{'Δnative':>9}")
rows = []
for tier in TIERS:
    m = measure(tier)
    rows.append(m)
    ratio = m["native_chars"] / m["dsl_chars"]
    rd, rn = REPORTED[tier]
    print(f"{tier:<14}{m['tools']:>6}{m['dsl_chars']:>7}{m['native_chars']:>8}{ratio:>6.2f}x   "
          f"{rd:>5}/{rn:<6}({rn/rd:.2f}x){m['dsl_chars']-rd:>+6}{m['native_chars']-rn:>+9}")

tier           tools    dsl  native  ratio   reported (ratio)      Δdsl  Δnative
iot_light_5        5   1334    2108  1.58x    1307/2062  (1.58x)   +27      +46
home_iot_20       20   2552    6842  2.68x    2525/6796  (2.69x)   +27      +46
smart_home_50     50   4670   15841  3.39x    4643/15795 (3.40x)   +27      +46


**절대값은 리포트와 다르다.** 세 tier 모두 DSL은 정확히 +27자, native는 정확히 +46자 커져 있다. 델타가 tier와 무관하게 상수라는 것은 **세 카탈로그가 공유하는 구성 요소 하나**(공통 도구의 description 등)가 리포트 작성 후에 바뀌었다는 뜻이다. `docs/factory_phase1_report.md`는 이미 `1,334 chars`로 기록하고 있으므로 변경은 2026-04-28과 05-08 사이에 일어났다. 비율은 1.58x / 2.68x / 3.39x로 리포트와 사실상 같다.

이 노트북이 하는 일이 바로 이런 것이다. 문서를 믿지 않고 코드로 다시 재고, 어긋나면 어디서 어긋났는지 적는다.

In [17]:
# 왜 DSL이 천천히 자라는가: 도구당 평균 문자 수
for m in rows:
    cat = TIERS[m["tier"]]
    n = len(cat.tools)
    # DSL은 헤더/규칙/예시가 고정비용이므로 도구 줄만 따로 센다
    tool_lines = [l for l in cat.render_json_dsl().splitlines() if l.startswith("- ")]
    dsl_per_tool = sum(len(l) for l in tool_lines) / n
    native_per_tool = m["native_chars"] / n
    print(f"{m['tier']:<14} tools={n:>2}  DSL/tool ≈ {dsl_per_tool:6.1f} chars   native/tool ≈ {native_per_tool:6.1f} chars")

iot_light_5    tools= 5  DSL/tool ≈  132.2 chars   native/tool ≈  421.6 chars
home_iot_20    tools=20  DSL/tool ≈   93.2 chars   native/tool ≈  342.1 chars
smart_home_50  tools=50  DSL/tool ≈   79.0 chars   native/tool ≈  316.8 chars


5 → 50으로 도구가 10배가 될 때 DSL은 약 3.5배(1,334 → 4,670), native는 약 7.5배(2,108 → 15,841) 커진다. DSL 쪽은 헤더·규칙·예시가 고정비용이라 도구 수에 비례하는 부분이 작고, native는 도구마다 JSON Schema 전체가 들어가므로 거의 선형이다. 이 **비율 확대**가 M2 실험의 핵심 가설이었고, 실제 Qwen API 토큰에서도 45% → 62.5% → 68.5% 절감으로 재현됐다(리포트 §14.4). 이 노트북은 API를 쓰지 않으므로 문자 수까지만 확인한다.

## 7. 재현 ②: 데이터셋 500건과 로드 시점 검증 (리포트 §6)

`examples/iot_light/dataset.jsonl`은 12건 seed + 488건 생성으로 이루어진 결정론적 데이터셋이다. 각 row는 `{"id","prompt","expected"}` 이고, **`expected`는 로드할 때 `parse_json_dsl`을 통과한다.** 즉 정답 자체가 Contract를 만족해야 하며, 정답이 깨져 있으면 데이터 로드 단계에서 실패한다 (`tests/test_dataset_integrity.py`가 이 성질을 지킨다).

In [18]:
from collections import Counter
from ganglion.benchmarks.iot.dataset import load_dataset

show(load_dataset)
cases = load_dataset()
print("cases:", len(cases))
print("first:", cases[0])

# /home/kist/workspace/ganglion/ganglion/benchmarks/iot/dataset.py
def load_dataset(path: Path = DEFAULT_DATASET, limit: int | None = None) -> list[EvalCase]:
    cases: list[EvalCase] = []
    with path.open(encoding="utf-8") as file:
        for line in file:
            if not line.strip():
                continue
            row: dict[str, Any] = json.loads(line)
            cases.append(
                EvalCase(
                    id=row["id"],
                    prompt=row["prompt"],
                    expected=parse_json_dsl(row["expected"]),
                )
            )
            if limit is not None and len(cases) >= limit:
                break
    return cases
cases: 500
first: EvalCase(id='iot-001', prompt='거실 불 70%로 켜줘', expected=ActionPlan(calls=(ToolCall(action='set_light', args={'room': 'living', 'state': 'on', 'brightness': 70}),)))


In [19]:
import re
_HANGUL = re.compile(r"[가-힣]")

action_counts = Counter(call.action for c in cases for call in c.expected.calls)
lang_counts = Counter("korean/mixed" if _HANGUL.search(c.prompt) else "english" for c in cases)

REPORTED_ACTIONS = {"set_light": 180, "schedule_light": 140, "get_light_state": 100,
                    "list_devices": 40, "create_scene": 40}
REPORTED_LANG = {"korean/mixed": 325, "english": 175}

print("action 분포 (리포트 §6 대비)")
for a, n in action_counts.most_common():
    print(f"  {a:<16}{n:>4}   reported {REPORTED_ACTIONS.get(a, '?'):>4}  {'✓' if REPORTED_ACTIONS.get(a) == n else '✗'}")
print("언어 분포")
for l, n in lang_counts.items():
    print(f"  {l:<16}{n:>4}   reported {REPORTED_LANG[l]:>4}  {'✓' if REPORTED_LANG[l] == n else '✗'}")

action 분포 (리포트 §6 대비)
  set_light        180   reported  180  ✓
  schedule_light   140   reported  140  ✓
  get_light_state  100   reported  100  ✓
  list_devices      40   reported   40  ✓
  create_scene      40   reported   40  ✓
언어 분포
  korean/mixed     325   reported  325  ✓
  english          175   reported  175  ✓


In [20]:
# expected가 정말 Contract를 통과하는지: 500건 전부 다시 파싱해 보고, ActionPlan 동등성도 확인
n_ok = 0
for c in cases:
    plan = iot.parse_json_dsl(c.expected.to_jsonable())
    assert plan == c.expected
    n_ok += 1
print(f"{n_ok}/{len(cases)} expected plans round-trip through parse_json_dsl and compare equal")

500/500 expected plans round-trip through parse_json_dsl and compare equal


## 8. 미리보기: 같은 Contract, 세 가지 확장

아래 세 기능은 각각 뒤 노트북의 주제다. 여기서는 "Contract 계층 안에 있다"는 사실과 진입점만 확인한다.

### 8.1 Null action — `allow_empty_calls` (BFCL M5, 8번 노트북)

In [21]:
from dataclasses import replace

iot_nocall = replace(iot, allow_empty_calls=True)
print("DSL 프롬프트에 추가되는 줄:")
print("  ", [l for l in iot_nocall.render_json_dsl().splitlines() if "calls\":[]" in l][0])
print("parse:", iot_nocall.parse_json_dsl('{"calls":[]}'))
try:
    iot.parse_json_dsl('{"calls":[]}')
except DSLValidationError as e:
    print("기본 카탈로그에서는:", e)

DSL 프롬프트에 추가되는 줄:
   If no tool call is needed, return exactly {"calls":[]}.
parse: ActionPlan(calls=())
기본 카탈로그에서는: 'calls' must not be empty


### 8.2 Post-correction — `prompt` 인자 (5번 노트북)

`parse_json_dsl(raw, prompt=...)` 에 사용자 프롬프트를 넘기면 `ToolSpec.defaults_when_missing`, `strip_unknown_args`, `prompt_correction` 이 validator **앞에서** 실행된다. 0.6B 모델의 실패 대부분이 이 층에서 회수됐다(86.4% → 99.2%). 지금은 세 규칙이 실제로 동작하는지만 본다.

In [22]:
demos = [
    # (규칙, raw, prompt)
    ("defaults_when_missing", "brightness만 있고 state 없음 → state='on'",
     '{"calls":[{"action":"set_light","args":{"room":"living","brightness":70}}]}',
     "거실 불 70%로 켜줘"),
    ("strip_unknown_args", "'#8'을 인자로 echo → 제거",
     '{"calls":[{"action":"list_devices","args":{"id":"8"}}]}',
     "조명 장치 목록 보여줘 #8"),
    ("prompt_correction", "오전 1시를 08:00으로 잘못 씀 → 01:00",
     '{"calls":[{"action":"schedule_light","args":{"room":"bedroom","at":"08:00","state":"off"}}]}',
     "오전 1시에 침실 조명 꺼줘"),
]
for rule, desc, raw, prompt in demos:
    print(f"[{rule}] {desc}")
    print("   prompt 없이 :", iot.parse_json_dsl(raw).calls[0].args)
    print("   prompt 주고 :", iot.parse_json_dsl(raw, prompt=prompt).calls[0].args)

[defaults_when_missing] brightness만 있고 state 없음 → state='on'
   prompt 없이 : {'room': 'living', 'state': 'on', 'brightness': 70}
   prompt 주고 : {'room': 'living', 'state': 'on', 'brightness': 70}
[strip_unknown_args] '#8'을 인자로 echo → 제거
   prompt 없이 : {}
   prompt 주고 : {}
[prompt_correction] 오전 1시를 08:00으로 잘못 씀 → 01:00
   prompt 없이 : {'room': 'bedroom', 'at': '08:00', 'state': 'off'}
   prompt 주고 : {'room': 'bedroom', 'at': '01:00', 'state': 'off'}


앞의 두 규칙은 `prompt` 없이도 동작한다. `defaults_when_missing`은 형제 인자(`brightness`)만 보고 결정하고, `strip_unknown_args`는 선언 목록만 보면 되기 때문이다. 세 번째 `prompt_correction`만 `prompt`가 있어야 발화한다. 프롬프트의 `오전 1시`를 읽어야 `08:00`이 틀렸다는 것을 알 수 있기 때문이다. 세 규칙이 validator 앞에서 어떤 순서로 실행되는지는 `validate_call`의 소스에 그대로 적혀 있다.

In [23]:
show(catalog_mod.Catalog.validate_call)

# /home/kist/workspace/ganglion/ganglion/contract/catalog.py
    def validate_call(
        self,
        raw_call: Any,
        depth: int,
        prompt: str | None = None,
    ) -> ToolCall:
        if not isinstance(raw_call, Mapping):
            raise DSLValidationError("each call must be an object")
        action = raw_call.get("action")
        if not isinstance(action, str):
            raise DSLValidationError("call.action must be a string")
        action = action.strip()
        tool = self.get_tool(action)
        if tool is None:
            raise DSLValidationError(f"unsupported action: {action}")
        args = raw_call.get("args", {})
        if not isinstance(args, Mapping):
            raise DSLValidationError("call.args must be an object")
        # Apply post-correction defaults BEFORE validation so a small model
        # that omits an obvious required arg (e.g. ``state`` when
        # ``brightness`` is set) doesn't fail validation. Rules only fire
        # wh

### 8.3 Schema compiler — 외부 스키마에서 Catalog 만들기 (8번 노트북)

BFCL, OpenAI, MCP 도구 스키마는 우리 `ToolSpec`으로 작성되어 있지 않다. `compile_tool_calling_schema`가 그것을 `Catalog`로 컴파일한다. BFCL은 케이스마다 도구 목록이 다르므로 **케이스당 Catalog 하나**를 만든다.

In [24]:
from ganglion.contract import compile_tool_calling_schema

bfcl_path = REPO / "examples/bfcl/v4/sample/simple_python.jsonl"
with bfcl_path.open(encoding="utf-8") as f:
    case = json.loads(f.readline())

print("question :", case["question"][0][0]["content"])
print("functions:", [fn["name"] for fn in case["function"]])
print("BFCL type names (non-standard):", case["function"][0]["parameters"]["type"])

mapper = compile_tool_calling_schema(case["function"], name=case["id"], allow_empty_calls=True)
print()
print(mapper.render_json_dsl())
plan = mapper.parse_json_dsl('{"calls":[{"action":"calculate_speed","args":{"distance":450,"time":20,"to_unit":"km/h"}}]}')
print("parsed :", plan)
print("emit   :", mapper.emit_tool_calls(plan))

question : Calculate the speed of an object in km/h if it traveled 450 meters in 20 seconds.
functions: ['calculate_speed']
BFCL type names (non-standard): dict

Return JSON only.
JSON shape: {"calls":[{"action":"<action>","args":{...}}]}
If no tool call is needed, return exactly {"calls":[]}.
Allowed actions:
- calculate_speed args {"distance": integer, "time": integer, "to_unit": optional string}
Rules:
- Do not include explanations or Markdown.

parsed : ActionPlan(calls=(ToolCall(action='calculate_speed', args={'distance': 450, 'time': 20, 'to_unit': 'km/h'}),))
emit   : [{'name': 'calculate_speed', 'arguments': {'distance': 450, 'time': 20, 'to_unit': 'km/h'}}]


BFCL의 `"type": "dict"` 같은 비표준 타입명이 컴파일러에서 `object`로 정규화된다. 그 다음부터는 §3~§4와 완전히 같은 `parse_json_dsl` / `ActionPlan` 경로다. 이것이 IoT와 BFCL이 **같은 validator, 같은 equality**로 채점되는 이유다.

### 8.4 Lenient 파싱 — `response_format` 없는 경로 (qwen-text / qwen-thinking)

structured output을 끄면 모델이 ```` ```json ```` 펜스나 설명문을 섞어 낼 수 있다. `parse_json_dsl_lenient`는 strict → fenced → embedded 순으로 구제하고, 어느 전략이 먹혔는지 돌려준다.

In [25]:
from ganglion.contract import parse_json_dsl_lenient

messy = '''Sure! Here is the plan:
```json
{"calls":[{"action":"get_light_state","args":{"room":"주방"}}]}
```
Let me know if you need anything else.'''
plan, strategy = parse_json_dsl_lenient(messy, catalog=iot)
print(strategy, "->", plan)

fenced -> ActionPlan(calls=(ToolCall(action='get_light_state', args={'room': 'kitchen'}),))


## 9. 정리

이 노트북에서 확인한 것:

1. **Contract는 leaf다.** `ToolSpec` 튜플 하나에서 DSL 프롬프트, OpenAI tools 스키마, validator, exact-match equality가 전부 파생된다. 이후 노트북의 어떤 숫자도 이 계층을 우회하지 않는다.
2. **정규화가 exact match의 정의다.** `aliases`, `bool_true`, `allow_percent`, `TimeArg` 덕에 모델 출력의 표면 변동이 흡수된다. 리포트의 "100%"는 이 정규화 **후** 수치다.
3. **카탈로그 크기 표 재현.** 5/20/50 tools에서 native/DSL = 1.58x / 2.69x / 3.40x. DSL은 고정비용 + 도구당 한 줄, native는 도구당 스키마 전체.
4. **데이터셋 500건 분포 재현.** action 180/140/100/40/40, 언어 325/175. `expected`는 로드 시 검증된다.
5. Contract 안의 세 확장(`allow_empty_calls`, post-correction, schema compiler)이 각각 BFCL M5, 0.6B 99.2%, BFCL 전이의 진입점이다.

### 연습 문제

- **B.** §1.1의 `toy` 카탈로그에 `TimeArg`를 쓰는 `schedule_fan` 도구를 추가하고, `"오후 3시"` 같은 입력이 왜 통과하지 못하는지 오류 메시지로 확인하라. 그 다음 `iot_light.py`의 `_correct_schedule_at`이 이 문제를 어떻게 푸는지 `show()`로 읽어 보라.
- **C.** `smart_home_50` 카탈로그에서 `render_json_dsl()`의 도구 줄 50개 중 가장 긴 줄과 가장 짧은 줄을 찾고, 그 도구의 native 스키마 길이와 비교하라. DSL/native 비율이 도구마다 얼마나 다른가?
- **D.** `ActionPlan` 동등성은 `calls` 순서에 민감하다. 두 호출의 순서만 바꾼 plan이 `!=`가 되는 것을 확인하고, BFCL `parallel` 카테고리에서 이것이 왜 문제가 되는지(리포트 §7 "parallel matching/count") 생각해 보라.

### 다음 노트북

**02 · Untuned Qwen3-0.6B 로컬 추론 → exact 38.6% 재현.** `ganglion/lm/local_hf.py`로 모델을 띄우고, 이 노트북의 `_dsl_messages`가 만든 프롬프트를 그대로 넣어 500건을 돌린다. 그 결과가 3번(합성)과 4번(SFT)의 baseline이 된다.